<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/proteomics-multiomics/lessons/P07-L02-imputation-and-batch-effects/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/proteomics-multiomics/lessons/P07-L02-imputation-and-batch-effects/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/proteomics-multiomics/lessons/P07-L02-imputation-and-batch-effects/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/proteomics-multiomics/lessons/P07-L02-imputation-and-batch-effects/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P07-L02 · Quantitation, missing values and batch effects: three imputations and a batch correction, counted against a planted truth

**You will build:** the analysis step that sits between a protein quantity table and a list of
"differentially abundant" proteins. You will write the Benjamini-Hochberg adjustment and a Welch
t-test that copes with missing values, three ways of filling those values in (a minimum value, the
values of the nearest neighbouring proteins in the same sample, and a draw from a normal fitted to
the low tail), and a location-scale batch correction. Then you will **count**, against a planted answer key, how many
proteins each choice calls at the same promised false discovery rate, how many of those calls are
right, and how much the call sets overlap.

**Time:** ~120 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download
· **Prerequisites:** `T00-L01-the-8gb-track` (the house shape of a lesson: stubs, public checks, an
autograded rubric) and `P07-L01-search-engine-and-fdr` (what a promised false discovery rate is a
promise about). Python and numpy indexing. No biology beyond "a protein has a quantity in each
sample, and the instrument sometimes fails to report it".

By the end you will be able to:

1. Implement the Benjamini-Hochberg adjustment and a Welch t-test that copes with missing values,
   and combine them into a differential-abundance call set at a promised false discovery rate.
2. Implement three imputation strategies for a label-free quantity matrix: a minimum value per
   sample, the nearest neighbouring proteins in the same sample, and a draw from a normal fitted to
   the low tail of each sample.
3. Implement a location-scale batch correction and measure what it does when batch is balanced
   against condition and when batch is correlated with condition.
4. Measure, against a planted answer key, how many proteins each imputation and each correction
   call, how many of those calls are truly differential, and how much the call sets overlap.
5. Explain, from the counts the notebook prints, why a list accepted at a promised false discovery
   rate can hold more false calls than promised, and why the answer to "which imputation is best"
   depends on which kind of missing value you have.

**The data is synthetic, and the simulator is in this notebook.** A real quantity table has no
answer key: nobody knows which proteins truly differ. A simulated one does, and a false discovery
rate is a claim about an answer key. `meta.yaml` declares the data as synthetic, nothing is
downloaded, and nothing you conclude about these particular counts describes a real instrument,
cohort or software package. What transfers is the method, and the habit of asking what a choice
did to the list.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T00-L01`, `P07-L01`

**This lesson assumes you can already:**

- write numpy code that handles missing values: np.isnan masks, nan-aware means and standard deviations, and row or column operations on a matrix
- say what a null hypothesis and a p-value are, and standardise a column of numbers to a z-score
- read a table of counts (called, truly differential, not) and compute a proportion from it

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Computer Age Statistical Inference, chapter 15: Large-Scale Hypothesis Testing and FDRs](https://hastie.su.domains/CASI/) · Bradley Efron and Trevor Hastie (Cambridge University Press; PDF free from the authors) · *free* — chapter 15 (section 15.2) sets out false discovery rates and the Benjamini-Hochberg procedure, and what a promised rate is a promise about
- [NIST/SEMATECH e-Handbook of Statistical Methods, 1.3.5.3 Two-Sample t-Test for Equal Means](https://www.itl.nist.gov/div898/handbook/eda/section3/eda353.htm) · NIST · *free* — gives the unequal-variance (Welch) statistic and the Welch-Satterthwaite degrees of freedom that the lesson's Welch test computes
- [Processing quantitative proteomics data with QFeatures](https://bioconductor.org/packages/release/bioc/vignettes/QFeatures/inst/doc/Processing.html) · Laurent Gatto (Bioconductor) · *free* — its 'Managing missing values' and 'Imputation' sections separate values missing at random from values missing because they fall below detection, the distinction the lesson turns on
- [A comparative study of evaluating missing value imputation methods in label-free proteomics](https://pmc.ncbi.nlm.nih.gov/articles/PMC7815892) · Jin et al. (Scientific Reports 2021, PubMed Central) · *free* — benchmarks seven imputation methods at varied shares of values missing not at random and reports false discoveries among altered proteins, the comparison the lesson reproduces in miniature
- [Tackling the widespread and critical impact of batch effects in high-throughput data](https://pmc.ncbi.nlm.nih.gov/articles/PMC3880143) · Leek et al. (Nature Reviews Genetics 2010, PubMed Central) · *free* — why a batch confounded with the condition of interest misleads, and the covariate-adjustment and surrogate-variable remedies, framing the balanced versus correlated batch designs

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import math
import os
import signal
import sys
import threading
import time
import traceback
from typing import Any, Callable, NamedTuple

import numpy as np

import matplotlib
_INTERACTIVE = "ipykernel" in sys.modules
if not _INTERACTIVE:
    # Headless: a script run (including this repository's execution gate) must never try to
    # open a window. In Jupyter the default inline backend is already the right one.
    matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402  (backend must be chosen before this import)

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__,
      "· matplotlib", matplotlib.__version__)

# The world. Every number that shapes it is here, so that nothing later is a hidden knob.
SEED = 20260930          # the world you study in detail; section 16 also uses the seeds after it
N_REPLICATES = 8
N_PROTEINS = 1200
N_PER_GROUP = 12         # samples per condition
FRAC_GRADED = 0.15       # proteins that differ by a moderate amount (a log2 fold change of 0.8 to 1.8)
FRAC_SWITCH = 0.04       # proteins that are on in one condition and off in the other (4 to 7 log2 units)
MU_MEAN, MU_SD = 24.0, 2.3   # a protein's typical log2 quantity is drawn from this normal
NOISE_SD = 0.40          # sample-to-sample noise, log2 units, before the protein-specific factor
DETECT_MID, DETECT_WIDTH = 21.5, 0.7   # a value at DETECT_MID is lost half the time, lower ones more often
MCAR_RATE = 0.04         # cells lost at random, whatever their value
BATCH_SHIFT, BATCH_SD, BATCH_SCALE = 0.3, 0.5, 1.3   # batch 1 vs batch 0: mean shift, per-protein shift, noise factor
MIN_OBSERVED = 4         # a protein reported in fewer than this many samples is dropped before analysis

# The analysis settings, fixed for the whole lesson so that only the choices you study vary.
ALPHA = 0.05             # the promised false discovery rate
K_NEIGHBOURS = 10
MIN_COMMON = 3
TAIL = 0.05


# How long an exercise's check may run before it counts as stuck, and how long any other cell that runs your
# code (a demonstration, a table) may: a right answer needs a small fraction of either, measured on the worked
# solution (internal/retrofit_guard.py, 10x or more). tools/verify_all.py reads both numbers.
_CHECK_SECONDS = 2
_DEMO_SECONDS = 60


class World(NamedTuple):
    """A simulated quantity matrix, and the answer key that generated it.

    `observed` is what an analyst receives: log2 quantities with NaN where nothing was reported.
    Everything after it is the answer key, which no real data set ever provides.
    """
    observed: np.ndarray       # (proteins, samples) float, NaN = not reported
    truth: np.ndarray          # the same matrix with nothing lost
    is_case: np.ndarray        # (samples,) bool: True = case, False = control
    batch: np.ndarray          # (samples,) int: 0 or 1
    is_de: np.ndarray          # (proteins,) bool: planted as truly different between conditions
    is_switch: np.ndarray      # (proteins,) bool: on/off proteins, a subset of is_de
    log2fc: np.ndarray         # (proteins,) planted case-minus-control difference, log2 units
    lost_low: np.ndarray       # (proteins, samples) bool: cell lost because its value was low
    lost_random: np.ndarray    # (proteins, samples) bool: cell lost at random, not already lost low


def generate_world(seed: int = SEED, confounded: bool = False, n_proteins: int = N_PROTEINS,
                   n_per_group: int = N_PER_GROUP) -> World:
    """Given to you. Simulate a label-free quantity matrix, two conditions, two batches.

    Each protein has a typical log2 quantity. Some proteins differ between conditions: most by a
    moderate amount, a few are "switch" proteins that are on in one condition and off in the other.
    Batch 1 adds a shift (the same for all proteins plus a protein-specific part) and more noise.
    A cell is lost when its value is low (a probability that rises as the value falls: missing not
    at random) or, independently, with a small fixed probability whatever its value (missing at
    random). `confounded=True` puts three quarters of the controls in batch 0 and three quarters of
    the cases in batch 1; otherwise each condition is split evenly across the batches.
    """
    rng = np.random.default_rng(seed)
    n = 2 * n_per_group
    is_case = np.arange(n) >= n_per_group
    if confounded:
        major = (3 * n_per_group) // 4
        batch = np.array([0] * major + [1] * (n_per_group - major)
                         + [0] * (n_per_group - major) + [1] * major)
    else:
        half = n_per_group // 2
        batch = np.array(([0] * half + [1] * (n_per_group - half)) * 2)
    mu = rng.normal(MU_MEAN, MU_SD, n_proteins)
    noise = NOISE_SD * np.exp(rng.normal(0.0, 0.25, n_proteins))
    kind = rng.random(n_proteins)
    is_switch = kind < FRAC_SWITCH
    graded = (kind >= FRAC_SWITCH) & (kind < FRAC_SWITCH + FRAC_GRADED)
    sign = np.where(rng.random(n_proteins) < 0.5, -1.0, 1.0)
    moderate = sign * rng.uniform(0.8, 1.8, n_proteins)
    big = sign * rng.uniform(4.0, 7.0, n_proteins)
    on_level = rng.normal(22.8, 1.0, n_proteins)
    log2fc = np.where(graded, moderate, np.where(is_switch, big, 0.0))
    # A switch protein sits at a level just around the detection limit where it is on.
    mu = np.where(is_switch, on_level - np.where(sign < 0, 0.0, big), mu)
    protein_shift = rng.normal(0.0, BATCH_SD, n_proteins)
    in_b1 = batch == 1
    truth = (mu[:, None] + log2fc[:, None] * is_case[None, :]
             + (BATCH_SHIFT + protein_shift)[:, None] * in_b1[None, :]
             + noise[:, None] * np.where(in_b1, BATCH_SCALE, 1.0)[None, :]
             * rng.standard_normal((n_proteins, n)))
    p_low = 1.0 / (1.0 + np.exp(-(DETECT_MID - truth) / DETECT_WIDTH))
    lost_low = rng.random(truth.shape) < p_low
    lost_random = (rng.random(truth.shape) < MCAR_RATE) & ~lost_low
    observed = np.where(lost_low | lost_random, np.nan, truth)
    return World(observed, truth, is_case, batch, graded | is_switch, is_switch, log2fc,
                 lost_low, lost_random)


def keep_observed(world: World, min_observed: int = MIN_OBSERVED) -> World:
    """Given to you. Drop proteins reported in fewer than `min_observed` samples, as every pipeline does."""
    keep = (~np.isnan(world.observed)).sum(axis=1) >= min_observed
    return World(world.observed[keep], world.truth[keep], world.is_case, world.batch,
                 world.is_de[keep], world.is_switch[keep], world.log2fc[keep],
                 world.lost_low[keep], world.lost_random[keep])


def _betacf(a: np.ndarray, b: float, x: np.ndarray) -> np.ndarray:
    """Continued fraction for the incomplete beta function (modified Lentz), on arrays."""
    tiny = 1e-300
    qab, qap, qam = a + b, a + 1.0, a - 1.0
    c = np.ones_like(x)
    d = 1.0 - qab * x / qap
    d = 1.0 / np.where(np.abs(d) < tiny, tiny, d)
    h = d.copy()
    for m in range(1, 501):
        m2 = 2 * m
        aa = m * (b - m) * x / ((qam + m2) * (a + m2))
        d = 1.0 + aa * d
        d = 1.0 / np.where(np.abs(d) < tiny, tiny, d)
        c = 1.0 + aa / c
        c = np.where(np.abs(c) < tiny, tiny, c)
        h = h * d * c
        aa = -(a + m) * (qab + m) * x / ((a + m2) * (qap + m2))
        d = 1.0 + aa * d
        d = 1.0 / np.where(np.abs(d) < tiny, tiny, d)
        c = 1.0 + aa / c
        c = np.where(np.abs(c) < tiny, tiny, c)
        delta = d * c
        h = h * delta
        if np.max(np.abs(delta - 1.0)) < 1e-15:
            break
    return h


_lgamma = np.vectorize(math.lgamma, otypes=[float])


def t_two_sided_p(t: np.ndarray, df: np.ndarray) -> np.ndarray:
    """Given to you. Two-sided p-value of Student's t with `df` degrees of freedom, on arrays.

    p = I_x(df/2, 1/2) with x = df / (df + t^2), the regularised incomplete beta function, evaluated
    by its continued fraction. NaN in, NaN out. A t of infinity gives 0.0 and a t of 0 gives 1.0.
    There is no scipy here, and a t-test needs this function: with a dozen samples the t
    distribution's heavy tails matter.
    """
    t = np.asarray(t, dtype=float)
    df = np.asarray(df, dtype=float)
    t, df = np.broadcast_arrays(t, df)
    out = np.full(t.shape, np.nan)
    ok = np.isfinite(df) & ~np.isnan(t) & (df > 0)
    x = np.where(ok, df / (df + np.where(np.isinf(t), 0.0, t) ** 2), 0.5)
    x = np.where(ok & np.isinf(t), 0.0, x)
    a = df / 2.0
    b = 0.5
    result = np.empty(t.shape)
    result[:] = np.nan
    edge0 = ok & (x <= 0.0)
    edge1 = ok & (x >= 1.0)
    mid = ok & ~edge0 & ~edge1
    result[edge0] = 0.0
    result[edge1] = 1.0
    if mid.any():
        am, xm = a[mid], x[mid]
        front = np.exp(_lgamma(am + b) - _lgamma(am) - math.lgamma(b)
                       + am * np.log(xm) + b * np.log1p(-xm))
        flip = xm > (am + 1.0) / (am + b + 2.0)
        res = np.empty(am.shape)
        if (~flip).any():
            res[~flip] = front[~flip] * _betacf(am[~flip], b, xm[~flip]) / am[~flip]
        if flip.any():
            res[flip] = 1.0 - front[flip] * _betacf(np.full(flip.sum(), b), am[flip],
                                                    1.0 - xm[flip]) / b
        result[mid] = res
    out[ok] = result[ok]
    return out


def _show(fig: "matplotlib.figure.Figure") -> None:
    """Display a figure in Jupyter, or close it cleanly in a headless script run."""
    if _INTERACTIVE:
        plt.show()
    else:
        plt.close(fig)


_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("benjamini_hochberg",),
    "exercise 2": ("welch_test",),
    "exercise 3": ("differential_calls",),
    "exercise 4": ("score_calls", "overlap_counts"),
    "exercise 5": ("missingness_by_intensity",),
    "exercise 6": ("impute_minimum",),
    "exercise 7": ("impute_knn",),
    "exercise 8": ("impute_left_tail",),
    "exercise 9": ("batch_correct",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run
_UPTO = {i: tuple(f"exercise {k}" for k in range(1, i + 1)) for i in range(1, 10)}
# The heavy demos (a whole world, then eight of them) run when the file is run as a notebook or a
# script, and are skipped when the autograder imports it, so grading stays quick.
_RUN_DEMOS = __name__ == "__main__"


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (differential_calls)"; several -> "exercises 3, 6 and 8"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


# A time limit for every check that calls the student's code: a loop that never ends must fail
# its check, not freeze the notebook or the grader (which imports the notebook). Copy this block
# unchanged; change only the message. Why it is this careful: whoever runs the notebook may
# already have an alarm running (tools/grade.py stops every rubric row after 45 s and repeats
# its alarm every 0.25 s; a harness may have an alarm that ends the process). Earlier guards
# switched that alarm off on the way out (`setitimer(ITIMER_REAL, 0)`), so the NEXT hang ran for
# ever. This one saves the running alarm, lets whichever deadline comes first win (handing the
# signal to the outer handler when it is the outer's turn), and on the way out re-arms the outer
# alarm with the time it had left and its interval. tools/verify_all.py fails a lesson whose
# guard does less.
class _OutOfTime(BaseException):
    """Raised inside the guarded block when its time is up. A BaseException, so `except
    Exception:` in the code under test cannot swallow it; it repeats every 0.25 s in case a bare
    `except:` does. `_time_limit` turns it into an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 2048):
    """Fail the block with an AssertionError naming `what` if it runs longer than `seconds`.
    Uses the operating system's alarm (Linux and macOS, main thread); a no-op elsewhere."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped block's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        import gc
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose stopping condition "
               "is never met runs for ever, or work grows far faster than the matrix does (a Python "
               "loop over every pair of cells instead of one array operation per row)")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_time_limit", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = (),
         limit: int = 0) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        with _time_limit(limit or (_CHECK_SECONDS if label in _EXERCISES else _DEMO_SECONDS), label):
            check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


def _demo(label: str, fn: Callable[[], None], needs: tuple[str, ...] = (), limit: int = 0) -> None:
    """A demo cell: guarded like a check, and skipped when the file is imported by the grader."""
    if _RUN_DEMOS:
        _try(label, fn, needs, limit)


def _given_check_t() -> None:
    """The one given helper that everything downstream trusts is checked against closed forms."""
    for tt in (0.5, 1.0, 2.0, 5.0):
        one = float(t_two_sided_p(np.array([tt]), np.array([1.0]))[0])
        two = float(t_two_sided_p(np.array([tt]), np.array([2.0]))[0])
        assert abs(one - (1.0 - 2.0 / math.pi * math.atan(tt))) < 1e-9, "t p-value, 1 df: broken"
        assert abs(two - (1.0 - tt / math.sqrt(2.0 + tt * tt))) < 1e-9, "t p-value, 2 df: broken"
    assert t_two_sided_p(np.array([0.0]), np.array([7.0]))[0] == 1.0, "t p-value: t = 0 must give p = 1"
    assert t_two_sided_p(np.array([np.inf]), np.array([7.0]))[0] == 0.0, "t p-value: t = inf must give p = 0"
    assert np.isnan(t_two_sided_p(np.array([np.nan]), np.array([7.0]))[0]), "t p-value: NaN in must give NaN out"
    print("the given t p-value function agrees with the closed forms for 1 and 2 degrees of freedom")


_try("given: t p-values", _given_check_t)

## 1. The problem in one page

In label-free quantitative proteomics the software reports a quantity for each protein in each
sample, here on a log2 scale. The result is a matrix with one row per protein and one column per
sample. It has holes: a protein that is scarce in a sample may
fall below what the instrument can detect, so no quantity is reported.

Two things about those holes decide everything that follows.

- **They are not all alike.** Some values are lost because the protein was too scarce: limits in
  detection drive values that are missing not at random, and the missing value tells you the
  quantity was *low*. Others are lost by chance, whatever the quantity: missing at random. A
  method aimed at one mechanism can be wrong for the other. That is the premise of the paper this
  lesson is built on: the question is not which imputation method is most accurate in general but
  which is most appropriate for the missing values at hand (`claims.yaml`).
- **A test needs numbers.** A t-test wants a full set of values per group. Imputation invents the
  missing ones, and then the test treats the invented values as if they had been measured.

A second problem is separate and just as important. Samples measured on different days, or on
different instruments, differ for reasons that are not biology: a **batch effect**. If batch is
spread evenly across the conditions you compare, it adds noise. If batch is *correlated* with the
condition, it can look exactly like a biological difference, and lead to incorrect conclusions
(`claims.yaml`).

The plan. Section 2 sets up the world. Sections 3 to 6 build the statistics, the test and the
scoring, and section 7 finds out what a complete matrix would give us. Sections 8 to 13 study the
missing values and the three imputations. Sections 14 and 15 add batch. Section 16 repeats
everything on several simulated worlds, so you can see how much of what you saw was the luck of
one seed.

## 2. The world you will study

`generate_world` (given, above) simulates the matrix and keeps the answer key beside it. Read what
it made before you analyse it. The cell below prints only what is computed from the world; nothing
in the prose is typed in from a run.

In [ ]:
WORLD = keep_observed(generate_world(SEED, confounded=False))
CONFOUNDED = keep_observed(generate_world(SEED, confounded=True))


def _describe_world(w: World, name: str) -> None:
    n_prot, n_samp = w.observed.shape
    print(f"{name}: {n_prot} proteins x {n_samp} samples "
          f"({int(w.is_case.sum())} case, {int((~w.is_case).sum())} control)")
    print("  samples by condition and batch:")
    for label, sel in (("control", ~w.is_case), ("case", w.is_case)):
        print(f"    {label:8s} batch 0: {int((sel & (w.batch == 0)).sum()):2d}   "
              f"batch 1: {int((sel & (w.batch == 1)).sum()):2d}")
    lost = np.isnan(w.observed)
    print(f"  cells not reported: {100 * lost.mean():.1f}%, of which "
          f"{100 * (lost & w.lost_low).sum() / lost.sum():.1f}% because the value was low and "
          f"{100 * (lost & w.lost_random).sum() / lost.sum():.1f}% at random")
    print(f"  planted truth: {int(w.is_de.sum())} proteins truly differ, "
          f"{int(w.is_switch.sum())} of them switch proteins (on in one condition, off in the other)")
    print(f"  proteins with every value reported: {int((~lost.any(axis=1)).sum())}")


_describe_world(WORLD, "balanced design")
print()
_describe_world(CONFOUNDED, "correlated design")

Two things to notice before you write any statistics. In the balanced design each condition is
split evenly over the two batches; in the correlated design most controls sit in one batch and most
cases in the other, so a difference between batches will look like a difference between conditions.
And compare the last line of each block with the number of proteins: an analysis that used only
complete rows would keep that many proteins and throw the rest away.

## 3. Exercise 1 — `benjamini_hochberg()`

Test many proteins and call every one whose p-value falls below the promised rate, and the
proteins that do not differ at all will still supply "hits": section 7 counts them. **False
discovery rate** control asks for something different: among the proteins you call, keep the
expected *fraction* that are false at or below a promised level. The
Benjamini-Hochberg procedure does that (`claims.yaml`). Given `m` p-values, sort them, and give
the *i*-th smallest the adjusted value `p_(i) * m / i`. Then make the adjusted values
non-decreasing in p by taking, for each one, the smallest adjusted value at or beyond it in the
sorted order (a hit is accepted at the lowest level that accepts anything at least as weak as it).
The result, the **q-value**, is the smallest promised rate at which that protein would be called.
You call every protein whose q-value is at most the promised rate.

You met the same running-minimum idea in `P07-L01-search-engine-and-fdr`, where the ratio was
decoys to targets; here the ratio comes from the p-values themselves.

<details><summary>💡 Hint 1 — what to think about</summary>

Work in the order of the sorted p-values, but give the answer back in the order you were given
them. What should happen to two p-values that are equal? They must end up with the same q-value
whatever order the sort placed them in. And what does the "smallest at or beyond" rule need: a
pass from the largest p-value downwards.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Sort to get an order and the ranks 1 to m. Scale each sorted p-value by m over its rank. Run a
running minimum from the last position back to the first. Put the results back in the original
positions. The call is `q <= alpha`, with the threshold itself included. Reject a p-value that is
not a number in the range 0 to 1, an input that is not one-dimensional, and an alpha that is not
above 0 and at most 1 (an alpha of 5 is a percentage, not a rate) with `ValueError` before
anything else.
</details>

In [ ]:
def benjamini_hochberg(p: np.ndarray, alpha: float = 0.05) -> tuple[np.ndarray, np.ndarray]:
    """Benjamini-Hochberg adjusted p-values (q-values) and the calls they give.

    For `m` p-values, the i-th smallest (rank i, counting from 1) gets `p * m / i`, and the
    q-value of each is the smallest such value at its own rank or any larger rank. Equal p-values
    get equal q-values. A protein is called when `q <= alpha`, the threshold included.

    Raise `ValueError` if `p` is not one-dimensional, holds anything that is not a number from 0 to
    1 (NaN included), or if `alpha` is not above 0 and at most 1.

    Example:
        >>> q, call = benjamini_hochberg(np.array([0.01, 0.04, 0.03, 0.005]), alpha=0.03)
        >>> q.round(6).tolist(), call.tolist()
        ([0.02, 0.04, 0.04, 0.02], [True, False, False, True])

    Returns:
        `(q, call)`: a float array and a bool array, both in the order of `p`; both empty when `p`
        is empty.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_bh() -> None:
    q, call = benjamini_hochberg(np.array([0.01, 0.04, 0.03, 0.005]), alpha=0.03)
    assert np.allclose(q, [0.02, 0.04, 0.04, 0.02]), (
        f"expected q = [0.02, 0.04, 0.04, 0.02], got {np.round(q, 6).tolist()}. Scale the i-th smallest "
        "p by m / i, then take a running minimum from the LARGEST p downwards, then restore the "
        "original order")
    assert call.tolist() == [True, False, False, True], (
        f"at alpha 0.03 the calls should be [True, False, False, True], got {call.tolist()}")
    q, _ = benjamini_hochberg(np.array([0.5, 0.01, 0.01]))
    assert np.allclose(q, [0.5, 0.015, 0.015]), (
        f"two equal p-values must get the same q-value; expected [0.5, 0.015, 0.015], got "
        f"{np.round(q, 6).tolist()}. Without the running minimum the first of the pair keeps a larger value")
    q, call = benjamini_hochberg(np.array([0.0625, 0.5, 0.25, 0.75]), alpha=0.25)
    assert call.tolist() == [True, False, False, False], (
        f"a q-value exactly equal to alpha IS called (<=): expected [True, False, False, False], got "
        f"{call.tolist()}")
    for bad_p, bad_a in ((np.array([0.2, np.nan]), 0.05), (np.array([0.2, 1.5]), 0.05),
                         (np.array([[0.2, 0.3]]), 0.05), (np.array([0.2]), 5.0),
                         (np.array([0.2]), 0.0)):
        try:
            benjamini_hochberg(bad_p, bad_a)
        except ValueError:
            pass
        else:
            raise AssertionError(f"benjamini_hochberg({bad_p!r}, {bad_a}) must raise ValueError")
    q, call = benjamini_hochberg(np.array([]))
    assert len(q) == 0 and len(call) == 0, "no p-values, no q-values and no calls"
    print("exercise 1 looks right — q-values from p-values, ties and the threshold handled")

In [ ]:
_try("exercise 1", _check_bh)

## 4. Exercise 2 — `welch_test()`

The usual test for "does this protein differ between conditions" compares the mean of the case
samples with the mean of the control samples, in units of their combined standard error. The
**Welch** version does not assume the two groups share a variance (`claims.yaml` gives the formulas):

- `log2fc` = mean(case) − mean(control), the case-minus-control difference on the log2 scale;
- `t` = log2fc / sqrt(v1/n1 + v0/n0), with `v` the sample variance (divide by n−1) and `n` the
  number of *reported* values in each group;
- `df` = (v1/n1 + v0/n0)² / [(v1/n1)²/(n1−1) + (v0/n0)²/(n0−1)];
- `p` = the two-sided p-value of `t` on `df` degrees of freedom: use the given `t_two_sided_p`.

Real matrices have holes, so this function must not fall over on them: it uses only the reported
values of each protein. A protein with fewer than two reported values in either group has no
variance to estimate and is **untestable**: all four outputs are NaN for it. Two edge cases matter
for what comes later. If every reported value in both groups is identical the standard error is
exactly zero: give `t` = 0 and `p` = 1 when the two means are equal, and `t` = ±infinity and
`p` = 0 when they are not.

<details><summary>💡 Hint 1 — what to think about</summary>

Count the reported values per row and per group first: those counts, not the number of columns,
are the n in every formula. A vector of NaN in a row should simply not take part. Which of numpy's
reductions ignore NaN, and which would let one hole spoil a whole row?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Work on the case columns and the control columns separately. For each, get the count, the mean of
the reported values and the sum of squared deviations from that mean over the reported values.
Mark the rows with fewer than two in either group, compute everything for all rows without letting
a division by zero raise, then overwrite the untestable rows with NaN and the zero-standard-error
rows with the special values. Reject an `is_case` whose length is not the number of columns, or
that leaves fewer than two case or fewer than two control columns, with `ValueError`.
</details>

In [ ]:
class TTest(NamedTuple):
    log2fc: np.ndarray
    t: np.ndarray
    df: np.ndarray
    p: np.ndarray


def welch_test(X: np.ndarray, is_case: np.ndarray) -> TTest:
    """Row-wise Welch t-test of case against control, using only the reported (non-NaN) values.

    `X` is (proteins, samples); `is_case` is a bool vector over the samples. For each row `n1` and
    `n0` are the reported values in the case and control samples. With fewer than 2 in either group
    the row is untestable and log2fc, t, df and p are all NaN. Otherwise log2fc, t and df are the
    formulas in the text (case minus control) and p is `t_two_sided_p(t, df)`. When the standard
    error is exactly zero: t = 0.0 and p = 1.0 if the means are equal, else t = +inf or -inf
    (the sign of the difference) and p = 0.0; df = n1 + n0 - 2 in both.

    Raise `ValueError` if `is_case` does not match the columns of `X`, or if fewer than two case
    or fewer than two control samples exist at all.

    Example:
        >>> X = np.array([[1.0, 2.0, 3.0, 3.0, 4.0, 5.0]])
        >>> r = welch_test(X, np.array([False, False, False, True, True, True]))
        >>> round(float(r.log2fc[0]), 3), round(float(r.t[0]), 3), round(float(r.df[0]), 3)
        (2.0, 2.449, 4.0)

    Returns:
        A `TTest` of four float arrays with one entry per row of `X`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_welch() -> None:
    case = np.array([False, False, False, True, True, True])
    r = welch_test(np.array([[1.0, 2.0, 3.0, 3.0, 4.0, 5.0]]), case)
    assert abs(r.log2fc[0] - 2.0) < 1e-9, (
        f"log2fc is mean(case) minus mean(control) = 2, got {r.log2fc[0]!r}: check the sign")
    assert abs(r.t[0] - 2.0 / math.sqrt(2.0 / 3.0)) < 1e-9 and abs(r.df[0] - 4.0) < 1e-9, (
        f"t should be 2 / sqrt(1/3 + 1/3) = 2.449 on 4 degrees of freedom, got t={r.t[0]!r}, df={r.df[0]!r}")
    assert 0.05 < r.p[0] < 0.1, f"a t of 2.449 on 4 degrees of freedom has a p-value between 0.05 and 0.1, got {r.p[0]!r}"
    r = welch_test(np.array([[1.0, 3.0, 10.0, 12.0, 14.0]]), np.array([False, False, True, True, True]))
    assert abs(r.df[0] - 2.882) < 0.01, (
        f"unequal group sizes and variances: the Welch degrees of freedom here are about 2.88, got "
        f"{r.df[0]!r}; the formula is (a + b)^2 / (a^2/(n1-1) + b^2/(n0-1)) with a = v1/n1, b = v0/n0")
    r = welch_test(np.array([[1.0, 2.0, np.nan, 3.0, 4.0, 5.0]]), case)
    assert abs(r.log2fc[0] - 2.5) < 1e-9 and abs(r.df[0] - 2.877) < 0.01, (
        f"with one control value missing the control mean is 1.5 and the case mean is 4, so log2fc is 2.5 "
        f"on about 2.88 degrees of freedom; got log2fc={r.log2fc[0]!r}, df={r.df[0]!r}. Use the reported "
        "values only, and let n be how many there are")
    r = welch_test(np.array([[1.0, np.nan, np.nan, 3.0, 4.0, 5.0], [np.nan] * 6]), case)
    assert np.isnan(r.log2fc[0]) and np.isnan(r.t[0]) and np.isnan(r.df[0]) and np.isnan(r.p[0]), (
        "one reported control value leaves no variance to estimate: the row is untestable and all four "
        "outputs are NaN")
    r = welch_test(np.array([[2.0, 2.0, 2.0, 4.0, 4.0, 4.0], [2.0, 2.0, 2.0, 2.0, 2.0, 2.0]]), case)
    assert r.t[0] == np.inf and r.p[0] == 0.0 and r.t[1] == 0.0 and r.p[1] == 1.0, (
        f"zero standard error: equal means give t=0 and p=1, different means give t=+inf and p=0; got "
        f"t={r.t.tolist()}, p={r.p.tolist()}")
    for bad in (np.array([False, True, True]), np.array([True, False, False, False, False, False])):
        try:
            welch_test(np.ones((1, 6)), bad)
        except ValueError:
            pass
        else:
            raise AssertionError("a design that does not fit the columns, or has one case sample, must raise ValueError")
    print("exercise 2 looks right — Welch t, degrees of freedom and p, holes skipped, edge cases handled")

In [ ]:
_try("exercise 2", _check_welch)

## 5. Exercise 3 — `differential_calls()`

Now put the two together. A call set needs three steps: test every protein that can be tested,
adjust *only those* p-values, and call the proteins whose q-value is at most the promised rate.
The middle step has a trap. An untestable protein has a NaN p-value and was never tested, so it
must not count towards `m` (the number of tests): counting it as a test with p = 1 makes every
other q-value larger and can quietly shrink the call set. Untestable proteins get a NaN q-value
and are never called.

The test and the correction are parameters, defaulting to your own `welch_test` and
`benjamini_hochberg`, so that later cells can swap either without touching this function.

<details><summary>💡 Hint 1 — what to think about</summary>

What tells you a protein was not tested? Which of the rows should be handed to the correction, and
which should be left out of it entirely but still appear, in their original place, in the output?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Run the test on the whole matrix. Build a mask of rows with a usable p-value. Send only those
p-values to the correction (it returns q-values and calls; use the q-values). Write them back into
an array of NaN of the full length. A row is called if it was testable and its q-value is at most
`alpha`.
</details>

In [ ]:
class Calls(NamedTuple):
    log2fc: np.ndarray    # case minus control, NaN if untestable
    p: np.ndarray         # NaN if untestable
    q: np.ndarray         # NaN if untestable
    called: np.ndarray    # bool: testable and q <= alpha


def differential_calls(X: np.ndarray, is_case: np.ndarray, alpha: float = ALPHA,
                       test: Callable | None = None, correct: Callable | None = None) -> Calls:
    """Test every testable protein, adjust those p-values, and call the proteins with q <= alpha.

    `test(X, is_case)` must return an object with `.log2fc` and `.p` (default `welch_test`);
    `correct(p, alpha)` must return `(q, call)` for a vector of p-values (default
    `benjamini_hochberg`). A row whose p is NaN is untestable: it is not passed to `correct`, its
    q is NaN, and it is not called.

    Example:
        >>> X = np.array([[1.0, 2.0, 3.0, 3.0, 4.0, 5.0], [1.0, np.nan, np.nan, 3.0, 4.0, 5.0]])
        >>> r = differential_calls(X, np.array([False] * 3 + [True] * 3))
        >>> np.isnan(r.q).tolist(), r.called.tolist()
        ([False, True], [False, False])

    Returns:
        A `Calls` of four arrays with one entry per row of `X`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_differential_calls() -> None:
    class _Fake(NamedTuple):
        log2fc: np.ndarray
        p: np.ndarray
    p = np.array([0.001, 0.02, np.nan, 0.9, 0.04])
    fake = lambda X, c: _Fake(np.arange(5, dtype=float), p)   # noqa: E731
    def bh(pv, alpha):   # an independent Benjamini-Hochberg for this one check
        order = np.argsort(pv)
        adj = np.minimum.accumulate((pv[order] * len(pv) / np.arange(1, len(pv) + 1))[::-1])[::-1]
        out = np.empty(len(pv))
        out[order] = adj
        return out, out <= alpha
    r = differential_calls(np.zeros((5, 4)), np.array([False, False, True, True]), alpha=0.045,
                           test=fake, correct=bh)
    assert np.isnan(r.q[2]) and not r.called[2], "an untestable row has a NaN q-value and is never called"
    assert r.called.tolist() == [True, True, False, False, False], (
        f"expected calls [True, True, False, False, False] at alpha 0.045, got {r.called.tolist()}. "
        "The NaN row must not count as a test: 4 p-values go to the correction, not 5")
    assert np.allclose(r.q[[0, 1, 3, 4]], [0.004, 0.04, 0.9, 0.05333333333], atol=1e-6), (
        f"q-values should come from the 4 testable p-values only, got {np.round(r.q, 6).tolist()}")
    r = differential_calls(np.array([[1.0, 2.0, 3.0, 3.0, 4.0, 5.0], [1.0, np.nan, np.nan, 3.0, 4.0, 5.0]]),
                           np.array([False] * 3 + [True] * 3))
    assert np.isnan(r.q).tolist() == [False, True] and r.called.tolist() == [False, False], (
        "with the real defaults the second row (one reported control value) is untestable and the first "
        f"is not called at this alpha: got q={r.q.tolist()}, called={r.called.tolist()}")
    print("exercise 3 looks right — only testable proteins are adjusted and called")

In [ ]:
_try("exercise 3", _check_differential_calls, needs=_UPTO[2])

## 6. Exercise 4 — `score_calls()` and `overlap_counts()`

This is the step a real analyst can never do. The planted truth says which proteins really differ,
so every call set can be counted:

- **called**: proteins in the list;
- **true calls**: called proteins that really differ; **false calls**: called proteins that do not;
- **missed**: proteins that really differ and were not called;
- **false discovery proportion** (`fdp`): false calls divided by called, or 0.0 when nothing was
  called. A promised rate is a claim about the *expected* value of this quantity.

Two call sets can have the same size and share nothing, so the second function counts how the
lists overlap: proteins only in the first, in both, and only in the second.

<details><summary>💡 Hint 1 — what to think about</summary>

Everything here is a count of an `and` of two masks. Which combination gives false calls, and which
gives missed proteins? What is the only case where the fdp needs a rule of its own?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Convert both inputs to bool arrays; reject two vectors of different lengths, or any that are not
one-dimensional, with `ValueError`. Sum each mask combination and turn the sums into plain
numbers. For the overlap, count `a and not b`, `a and b` and `b and not a`.
</details>

In [ ]:
class CallScore(NamedTuple):
    called: int
    true_calls: int
    false_calls: int
    missed: int
    fdp: float


def score_calls(called: np.ndarray, truth: np.ndarray) -> CallScore:
    """Count a call set against the planted truth.

    `called` and `truth` are bool vectors over the proteins. `fdp` is `false_calls / called`, and
    0.0 when nothing was called. Raise `ValueError` if the two are not one-dimensional and of the
    same length.

    Example:
        >>> score_calls(np.array([True, True, False, False, True]), np.array([True, False, True, False, True]))
        CallScore(called=3, true_calls=2, false_calls=1, missed=1, fdp=0.3333333333333333)

    Returns:
        A `CallScore`: four integers and a float.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def overlap_counts(a: np.ndarray, b: np.ndarray) -> tuple[int, int, int]:
    """How two call sets overlap: `(only_a, both, only_b)`.

    Raise `ValueError` if `a` and `b` are not one-dimensional and of the same length.

    Example:
        >>> overlap_counts(np.array([True, True, False, False]), np.array([True, False, True, False]))
        (1, 1, 1)

    Returns:
        Three integers.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_scoring() -> None:
    s = score_calls(np.array([True, True, False, False, True]), np.array([True, False, True, False, True]))
    assert (s.called, s.true_calls, s.false_calls, s.missed) == (3, 2, 1, 1), (
        f"expected (called, true, false, missed) = (3, 2, 1, 1), got {tuple(s[:4])!r}. A false call is "
        "called AND not truly different; a missed protein is truly different AND not called")
    assert abs(s.fdp - 1 / 3) < 1e-12, f"fdp is false calls over called = 1/3, got {s.fdp!r}"
    s = score_calls(np.zeros(4, dtype=bool), np.array([True, False, False, True]))
    assert s.fdp == 0.0 and s.missed == 2 and s.called == 0, (
        f"a call set with nothing in it has fdp 0.0 (not an error, not NaN) and misses every true "
        f"protein: got {s!r}")
    assert overlap_counts(np.array([True, True, False, False]), np.array([True, False, True, False])) == (1, 1, 1), (
        "overlap_counts returns (only in a, in both, only in b)")
    assert overlap_counts(np.array([True, True, True, False]), np.array([True, False, False, False])) == (2, 1, 0), (
        "the order matters: (only in a, in both, only in b)")
    for f in (score_calls, overlap_counts):
        try:
            f(np.zeros(3, dtype=bool), np.zeros(4, dtype=bool))
        except ValueError:
            pass
        else:
            raise AssertionError(f"{f.__name__} of vectors with different lengths must raise ValueError")
    print("exercise 4 looks right — call sets counted against the answer key")

In [ ]:
_try("exercise 4", _check_scoring)

## 7. The best case: no values missing at all

Before any value goes missing, find out what your machinery does on the complete matrix, `truth`,
which only a simulation can hand you. This is the ceiling for everything that follows, and a
calibration of the promise: at the promised rate, is the false discovery proportion where it was
promised to be? The second row shows what happens without the adjustment, calling every protein whose
p-value is below the promised rate.

In [ ]:
def _the_complete_matrix() -> None:
    w = WORLD
    res = differential_calls(w.truth, w.is_case, ALPHA)
    adjusted = score_calls(res.called, w.is_de)
    unadjusted = score_calls(res.p < ALPHA, w.is_de)
    print(f"{int((~w.is_de).sum())} of the {len(w.is_de)} proteins are planted as not differing, "
          f"{int(w.is_de.sum())} as differing; promised false discovery rate {ALPHA}")
    print(f"{'':28s} {'called':>7s} {'true':>6s} {'false':>6s} {'missed':>7s} {'fdp':>7s}")
    for name, s in (("Benjamini-Hochberg", adjusted), (f"raw p < {ALPHA}, no adjustment", unadjusted)):
        print(f"{name:28s} {s.called:7d} {s.true_calls:6d} {s.false_calls:6d} {s.missed:7d} "
              f"{100 * s.fdp:6.1f}%")
    where = "at or below" if adjusted.fdp <= ALPHA else "above"
    print(f"on this matrix the adjusted list's false discovery proportion, {100 * adjusted.fdp:.1f}%, "
          f"is {where} the promised {100 * ALPHA:.1f}%. The promise is about an expectation over many "
          "matrices, so one matrix can sit on either side; section 16 repeats this on several.")
    print(f"the unadjusted list holds {unadjusted.false_calls - adjusted.false_calls:+d} false calls "
          "relative to the adjusted one")


_demo("the complete matrix", _the_complete_matrix, needs=_UPTO[4])

## 8. Exercise 5 — `missingness_by_intensity()`

Which kind of missing value do we have? A real analyst has no answer key, but there is a
symptom. If values are lost because they are low, proteins that are low on average lose *more*
values than proteins that are high. If values are lost at random, every level of abundance loses
about the same share.

Rank the proteins by the mean of their reported values, split the ranking into `n_bins` groups of
as-equal-as-possible size (the lowest-mean proteins first; when the sizes cannot be equal, the
earlier groups take the extra proteins, as `np.array_split` does), and report, per group, the
fraction of all its cells that are not reported. Rank ties keep the original protein order.

<details><summary>💡 Hint 1 — what to think about</summary>

The mean has to be taken over what was reported, or a protein with holes would look lower than it
is. What has no mean at all, and what should the function do about it rather than return NaN?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Get the mean of the reported values per row. Order the rows by it with a sort that keeps the
original order among equals. Cut the ordered row numbers into consecutive groups. For each group
take the share of NaN among all of its cells. Raise `ValueError` for a row with nothing reported,
for a matrix that is not two-dimensional, and for `n_bins` that is not a whole number from 1 up to
the number of rows.
</details>

In [ ]:
def missingness_by_intensity(X: np.ndarray, n_bins: int = 5) -> np.ndarray:
    """Fraction of cells not reported, in `n_bins` groups of proteins ranked by mean reported value.

    Rows are ordered by the mean of their reported values, lowest first (a stable sort, so equal
    means keep their row order), and cut into `n_bins` consecutive groups exactly as
    `np.array_split` would cut the row numbers. Each entry is the share of NaN among all the cells
    of the rows in that group.

    Raise `ValueError` if `X` is not 2-D, if any row has no reported value, or if `n_bins` is not
    an integer from 1 to the number of rows.

    Example:
        >>> X = np.array([[10.0, np.nan], [1.0, np.nan], [5.0, 7.0], [4.0, np.nan]])
        >>> missingness_by_intensity(X, n_bins=2).tolist()
        [0.5, 0.25]

    Returns:
        A float array of length `n_bins`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_missingness() -> None:
    X = np.array([[10.0, np.nan], [1.0, np.nan], [5.0, 7.0], [4.0, np.nan]])
    got = missingness_by_intensity(X, n_bins=2)
    assert np.allclose(got, [0.5, 0.25]), (
        f"expected [0.5, 0.25], got {np.round(got, 4).tolist()}. Rank by the mean of the REPORTED values, "
        "lowest first, and report the share of NaN cells in each group")
    ranks = [3, 8, 0, 5, 9, 1, 7, 4, 2, 6]      # the row order is scrambled on purpose
    X = np.array([[float(r), np.nan if r in (0, 5) else float(r)] for r in ranks])
    got = missingness_by_intensity(X, n_bins=4)
    assert np.allclose(got, [1 / 6, 1 / 6, 0.0, 0.0]), (
        f"ten rows in four groups are cut 3, 3, 2, 2 in order of the mean (earlier groups take the extra "
        f"rows); expected [1/6, 1/6, 0, 0] = [0.1667, 0.1667, 0, 0], got {np.round(got, 4).tolist()}")
    got = missingness_by_intensity(np.array([[1.0, np.nan], [2.0, 2.0], [np.nan, 3.0]]), n_bins=3)
    assert np.allclose(got, [0.5, 0.0, 0.5]), f"one row per group: expected [0.5, 0, 0.5], got {got.tolist()}"
    for bad_x, bad_b in ((np.array([[np.nan, np.nan], [1.0, 2.0]]), 1), (np.ones((3, 2)), 0),
                         (np.ones((3, 2)), 4), (np.ones((3, 2)), 1.5), (np.ones(3), 1)):
        try:
            missingness_by_intensity(bad_x, bad_b)
        except ValueError:
            pass
        else:
            raise AssertionError(f"missingness_by_intensity({bad_x.shape}, {bad_b}) must raise ValueError")
    print("exercise 5 looks right — the missing share by abundance group")

In [ ]:
_try("exercise 5", _check_missingness)

In [ ]:
def _the_missingness_signature() -> None:
    w, n_bins = WORLD, 5
    frac = missingness_by_intensity(w.observed, n_bins)
    means = np.nansum(w.observed, axis=1) / (~np.isnan(w.observed)).sum(axis=1)
    order = np.argsort(means, kind="stable")
    lost = np.isnan(w.observed)
    print(f"{'group':>5s} {'mean of reported values':>24s} {'cells not reported':>19s} "
          f"{'of which lost because low':>27s}")
    for k, idx in enumerate(np.array_split(order, n_bins)):
        share_low = (lost[idx] & w.lost_low[idx]).sum() / max(lost[idx].sum(), 1)
        print(f"{k + 1:5d} {means[idx].mean():24.2f} {100 * frac[k]:18.1f}% {100 * share_low:26.1f}%")
    falling = bool(np.all(np.diff(frac) <= 0))
    print(f"the share of cells not reported {'falls at every step' if falling else 'does not fall at every step'} "
          f"as the mean rises; the lowest group loses {frac[0] / max(frac[-1], 1e-12):.1f} times the "
          "share the highest group does.")
    print("The last column uses the answer key, which a real analyst does not have: if the "
          "share lost because low is large where the fraction missing is large, the symptom you can "
          "see is pointing at the mechanism you cannot.")
    fig, ax = plt.subplots(figsize=(5.5, 3.2))
    ax.bar(np.arange(1, n_bins + 1), 100 * frac)
    ax.set_xlabel("proteins ranked by mean reported value (1 = lowest)")
    ax.set_ylabel("cells not reported, %")
    fig.tight_layout()
    _show(fig)


_demo("missingness signature", _the_missingness_signature, needs=_UPTO[5])

## 9. Exercise 6 — `impute_minimum()`

A common answer to a low value that is missing is to put a low value there. **Minimum-value
imputation** replaces every missing cell in a sample with a low quantile of that sample's reported
values: with `q = 0`, its smallest reported value. It is done per sample (per column) because
samples differ in overall level. It is quick, and the reasoning is that a missing value here means
"below what was detected". Whether that reasoning holds for a particular missing cell is exactly
what sections 12 and 13 measure.

The quantile is `np.quantile` with its default (linear) interpolation. Reject a `q` outside 0 to
1, and a column with no reported value at all, with `ValueError`. Return a new array; do not touch
the input, and do not touch a reported value.

<details><summary>💡 Hint 1 — what to think about</summary>

Per column: which cells are reported, and which are holes? What quantile of the reported cells do
you fill the holes with, and does it change from cell to cell within a column?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Copy the matrix. For each column take the reported values, compute the requested quantile of just
those, and write it into that column's NaN cells. Check `q` and each column's count before you fill.
</details>

In [ ]:
def impute_minimum(X: np.ndarray, q: float = 0.0) -> np.ndarray:
    """Fill each NaN with the `q` quantile of the reported values in its own column.

    `q = 0` is the column minimum. Uses `np.quantile` with its default linear interpolation on the
    reported values of that column. Reported values are never changed and `X` is not modified.
    Raise `ValueError` if `q` is outside 0 to 1 or a column has no reported value.

    Example:
        >>> X = np.array([[1.0, np.nan], [3.0, 5.0], [np.nan, 9.0]])
        >>> impute_minimum(X).tolist()
        [[1.0, 5.0], [3.0, 5.0], [1.0, 9.0]]

    Returns:
        A new float array with no NaN.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_minimum() -> None:
    X = np.array([[1.0, np.nan], [3.0, 5.0], [np.nan, 9.0]])
    got = impute_minimum(X)
    assert got.tolist() == [[1.0, 5.0], [3.0, 5.0], [1.0, 9.0]], (
        f"each hole takes the minimum of ITS OWN column: expected [[1, 5], [3, 5], [1, 9]], got {got.tolist()}")
    assert np.isnan(X[0, 1]), "the input matrix must not be modified"
    X = np.array([[np.nan, 4.0], [8.0, np.nan], [2.0, 6.0], [6.0, 2.0], [4.0, 8.0]])
    got = impute_minimum(X, q=0.25)
    assert abs(got[0, 0] - 3.5) < 1e-12 and abs(got[1, 1] - 3.5) < 1e-12, (
        f"q = 0.25 of the reported values [8, 2, 6, 4] is 3.5 (linear interpolation), got {got[0, 0]!r} and {got[1, 1]!r}")
    assert got[2, 0] == 2.0 and got[4, 1] == 8.0, "reported values must stay exactly as they were"
    for bad_x, bad_q in ((X, -0.1), (X, 1.1), (np.array([[np.nan, 1.0], [np.nan, 2.0]]), 0.0)):
        try:
            impute_minimum(bad_x, bad_q)
        except ValueError:
            pass
        else:
            raise AssertionError("q outside 0 to 1, or a column with nothing reported, must raise ValueError")
    print("exercise 6 looks right — holes filled with a low value of their own sample")

In [ ]:
_try("exercise 6", _check_minimum)

## 10. Exercise 7 — `impute_knn()`

The opposite philosophy fills a hole with what *similar proteins* did in that sample
(the k-nearest-neighbour approach that was proposed for microarray data, `claims.yaml`). The rule
in this lesson is exact, so that two correct programs give the same numbers:

- the **distance** between two proteins is the root of the mean squared difference over the samples
  in which *both* were reported. Two proteins with fewer than `min_common` such samples are not
  comparable;
- to fill the hole of protein *i* in sample *j*, take the proteins that are comparable to *i* and
  *were reported in sample j*, and average their values in sample *j* for the `k` nearest of them
  (a tie in distance goes to the protein with the lower row number; fewer than `k` candidates
  means use all of them);
- only originally reported values are ever used, never a value this function has already filled;
- if there is no candidate at all, use the mean of the reported values in column *j*.

Written cell by cell in plain Python this is far too slow for a full matrix. Distances from one
protein to every other can be computed in one array operation.

<details><summary>💡 Hint 1 — what to think about</summary>

Which quantity decides who is nearest: the total squared difference, or its mean over the samples
both proteins share? What if one candidate has only three samples in common and another has
twenty? And which proteins can be neighbours for a given hole: a candidate must have a value in
*that* sample.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Loop over the proteins that have a hole. For one of them, get a mask of the samples it shares with
every other protein and use it to compute all the distances in one go, with holes replaced by a
placeholder that the mask then ignores. Mark who is comparable (enough shared samples, not
itself). For each hole of that protein, pick the comparable proteins reported in that sample,
sort them by distance with a sort that keeps the row order among ties, keep the first `k`, and
average their original values. Reject `k` or `min_common` below 1, and a column with nothing
reported, with `ValueError`.
</details>

In [ ]:
def impute_knn(X: np.ndarray, k: int = K_NEIGHBOURS, min_common: int = MIN_COMMON) -> np.ndarray:
    """Fill each NaN from the `k` most similar proteins that were reported in that sample.

    distance(i, r) = sqrt(mean over the samples where both i and r are reported of (X[i] - X[r])^2);
    protein r is a candidate for the hole (i, j) if r != i, X[r, j] is reported and i and r share at
    least `min_common` samples. The fill is the mean of X[r, j] over the `k` candidates with the
    smallest distance (ties: lower row number first; fewer than `k` candidates: use them all). Only
    reported values are used. With no candidate the fill is the mean of the reported values in
    column j. Reported values are never changed and `X` is not modified.

    Raise `ValueError` if `k < 1`, `min_common < 1` or a column has no reported value.

    Example:
        >>> X = np.array([[1.0, 2.0, np.nan], [1.0, 2.0, 10.0], [9.0, 8.0, 50.0], [1.0, 2.5, 20.0]])
        >>> impute_knn(X, k=2, min_common=2)[0, 2]
        15.0

    Returns:
        A new float array with no NaN.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_knn() -> None:
    X = np.array([[1.0, 2.0, np.nan], [1.0, 2.0, 10.0], [9.0, 8.0, 50.0], [1.0, 2.5, 20.0]])
    got = impute_knn(X, k=2, min_common=2)
    assert abs(got[0, 2] - 15.0) < 1e-12, (
        f"protein 0's two nearest neighbours reported in sample 2 are proteins 1 and 3 (values 10 and 20), "
        f"so the fill is 15; got {got[0, 2]!r}")
    assert np.isnan(X[0, 2]) and not np.isnan(got).any(), "return a new, complete array and leave the input alone"
    # The mean over shared samples, not the sum: protein 1 shares two samples with protein 0 (a small
    # sum of squares, a large mean), protein 2 shares four (a larger sum, a smaller mean).
    X = np.array([[0.0, 0.0, 0.0, 0.0, np.nan],
                  [1.0, 1.0, np.nan, np.nan, 100.0],
                  [0.9, 0.9, 0.9, 0.9, 200.0]])
    got = impute_knn(X, k=1, min_common=2)
    assert got[0, 4] == 200.0, (
        f"distance is the root of the MEAN squared difference over shared samples: protein 2 (mean square "
        f"0.81) is nearer than protein 1 (mean square 1) although its sum of squares is larger; expected the "
        f"fill 200, got {got[0, 4]!r}")
    got = impute_knn(X, k=1, min_common=3)
    assert got[0, 4] == 200.0, "min_common=3 leaves only protein 2 comparable"
    got = impute_knn(X, k=1, min_common=5)
    assert abs(got[0, 4] - 150.0) < 1e-12, (
        f"with no comparable candidate the fill is the mean of the reported values in the column (100 and 200): "
        f"expected 150, got {got[0, 4]!r}")
    X = np.array([[0.0, 0.0, np.nan], [1.0, 1.0, 10.0], [-1.0, -1.0, 30.0], [5.0, 5.0, 90.0]])
    got = impute_knn(X, k=1, min_common=2)
    assert got[0, 2] == 10.0, f"a tie in distance goes to the lower row number; expected 10, got {got[0, 2]!r}"
    X = np.array([[0.0, np.nan], [0.4, np.nan], [-0.5, 10.0], [0.9, 30.0]])
    got = impute_knn(X, k=1, min_common=1)
    assert got[0, 1] == 10.0 and got[1, 1] == 30.0, (
        f"holes are filled from ORIGINAL values only, never from a value already filled in: protein 1's nearest "
        f"protein that was really reported in that sample is protein 3, so expected the fills [10, 30], got "
        f"{got[:2, 1].tolist()}")
    for kk, mc in ((0, 3), (3, 0)):
        try:
            impute_knn(np.array([[1.0, np.nan], [2.0, 3.0]]), kk, mc)
        except ValueError:
            pass
        else:
            raise AssertionError("k or min_common below 1 must raise ValueError")
    print("exercise 7 looks right — holes filled from the nearest proteins reported in that sample")

In [ ]:
_try("exercise 7", _check_knn)

## 11. Exercise 8 — `impute_left_tail()`

The third strategy takes the "low value" reasoning seriously and adds spread. For each sample,
look at the low end of what was reported: the reported values at or below the `tail` quantile of
that sample. Fit a normal distribution to them (their mean and their sample standard deviation,
dividing by n−1) and fill each hole in the sample with a draw from that normal. A hole is a low
value, so it is drawn from where the low values are, and the draws differ from each other instead
of all being the same constant.

To make two correct programs give the same numbers, the random draws are supplied: `z` is a matrix
of standard-normal numbers of the same shape as `X`, and the fill for the hole `(i, j)` is
`mean_j + sd_j * z[i, j]`. Reject a `tail` outside the range above 0 and up to 1, a `z` of another
shape, a sample with nothing reported, and a sample with fewer than two values in its low end,
with `ValueError`.

<details><summary>💡 Hint 1 — what to think about</summary>

Everything is per sample. Which values make up the low end: those *below* the quantile or those
*at or below* it? When the quantile lands exactly on a reported value, that value is one of the
low values. What do you need from them to build a normal draw?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

For each column take the reported values, find their `tail` quantile with numpy's default
interpolation, keep the reported values at or below it, and take the mean and the sample standard
deviation of those. Put mean plus standard deviation times the matching entry of `z` into each NaN
cell of that column. Reported cells stay as they were.
</details>

In [ ]:
def impute_left_tail(X: np.ndarray, z: np.ndarray, tail: float = TAIL) -> np.ndarray:
    """Fill each NaN with a draw from a normal fitted to the low end of its own column.

    For column j: `cut` is `np.quantile` (default interpolation) of the reported values at `tail`;
    the low end is the reported values `<= cut`; `mean_j` and `sd_j` are their mean and their
    standard deviation with n - 1 in the denominator. The fill for a NaN at (i, j) is
    `mean_j + sd_j * z[i, j]`. Reported values are never changed and `X` is not modified.

    Raise `ValueError` if `tail` is not above 0 and at most 1, if `z` does not have the shape of
    `X`, if a column has no reported value, or if a column's low end has fewer than 2 values.

    Example:
        >>> X = np.array([[1.0], [2.0], [3.0], [4.0], [5.0], [np.nan]])
        >>> round(float(impute_left_tail(X, np.ones((6, 1)), tail=0.4)[5, 0]), 4)
        2.2071

    Returns:
        A new float array with no NaN.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_left_tail() -> None:
    X = np.array([[1.0], [2.0], [3.0], [4.0], [5.0], [np.nan]])
    got = impute_left_tail(X, np.ones((6, 1)), tail=0.4)
    assert abs(got[5, 0] - (1.5 + math.sqrt(0.5))) < 1e-9, (
        f"the 0.4 quantile of [1, 2, 3, 4, 5] is 2.6, so the low end is [1, 2]: mean 1.5 and standard deviation "
        f"sqrt(0.5) with n - 1 in the denominator; the fill for z = 1 is 1.5 + sqrt(0.5) = 2.2071, got {got[5, 0]!r}")
    X = np.array([[1.0, 5.0], [2.0, np.nan], [3.0, 9.0], [np.nan, 7.0], [4.0, 6.0], [5.0, 8.0], [6.0, 10.0],
                  [7.0, 11.0], [8.0, 12.0]])
    z = np.arange(18, dtype=float).reshape(9, 2) / 10.0
    got = impute_left_tail(X, z, tail=0.25)
    # column 0: reported 1..8 (without the NaN row 3): quantile 0.25 of [1,2,3,4,5,6,7,8] is 2.75 -> low end [1, 2]
    lo0 = np.array([1.0, 2.0])
    assert abs(got[3, 0] - (lo0.mean() + lo0.std(ddof=1) * z[3, 0])) < 1e-9, (
        f"fitted per column, the low end of column 0 is [1, 2]; the fill uses z[3, 0]; got {got[3, 0]!r}")
    # column 1: reported [5,9,7,6,8,10,11,12] -> quantile 0.25 (linear) = 6.75 -> low end [5, 6]
    lo1 = np.array([5.0, 6.0])
    assert abs(got[1, 1] - (lo1.mean() + lo1.std(ddof=1) * z[1, 1])) < 1e-9, (
        f"each column has its own low end: column 1's is [5, 6]; got {got[1, 1]!r}")
    assert got[0, 0] == 1.0 and got[6, 1] == 10.0, "reported values must stay exactly as they were"
    X = np.array([[1.0], [2.0], [3.0], [4.0], [5.0], [6.0], [7.0], [8.0], [9.0], [np.nan]])
    got = impute_left_tail(X, np.zeros((10, 1)), tail=0.25)
    assert abs(got[9, 0] - 2.0) < 1e-9, (
        f"the 0.25 quantile of nine values 1..9 is exactly 3, and a value equal to the cut is part of the low "
        f"end: [1, 2, 3] has mean 2; expected the fill 2 at z = 0, got {got[9, 0]!r}")
    Xb = np.array([[1.0], [2.0], [3.0], [np.nan]])
    for what, args in (("a tail of 0", (Xb, np.ones((4, 1)), 0.0)), ("a tail above 1", (Xb, np.ones((4, 1)), 1.5)),
                       ("a z of the wrong shape", (Xb, np.ones((3, 1)), 0.5))):
        try:
            impute_left_tail(*args)
        except ValueError:
            pass
        else:
            raise AssertionError(f"impute_left_tail with {what} must raise ValueError")
    try:
        impute_left_tail(np.array([[1.0], [2.0], [np.nan]]), np.ones((3, 1)), 0.1)
    except ValueError:
        pass
    else:
        raise AssertionError("a low end with fewer than two values has no standard deviation: ValueError")
    print("exercise 8 looks right — holes drawn from a normal fitted to the low end of each sample")

In [ ]:
_try("exercise 8", _check_left_tail)

## 12. What each imputation does to the missing cells

Before counting call sets, look at the imputed values themselves. The answer key holds the value
each lost cell would have had, so the error of every fill can be measured, separately for the
cells lost because they were low and the cells lost at random. `draws` are the standard-normal
numbers your `impute_left_tail` consumes; they are generated once, from a fixed seed.

In [ ]:
STRATEGIES = ("no imputation", "minimum", "nearest neighbours", "left tail")


def draws_for(world: World, seed: int) -> np.ndarray:
    """Given to you. The standard-normal draws `impute_left_tail` uses, fixed by the seed."""
    return np.random.default_rng(seed + 1000).standard_normal(world.observed.shape)


def impute_all(world: World, seed: int) -> dict[str, np.ndarray]:
    """Given to you. Each strategy applied to the observed matrix, with your three functions."""
    X = world.observed
    return {"no imputation": X,
            "minimum": impute_minimum(X),
            "nearest neighbours": impute_knn(X, K_NEIGHBOURS, MIN_COMMON),
            "left tail": impute_left_tail(X, draws_for(world, seed), TAIL)}


def _the_imputers() -> None:
    w = WORLD
    imputed = impute_all(w, SEED)
    seen = ~np.isnan(w.observed)
    print(f"{'strategy':20s} {'cells lost because low':^28s} {'cells lost at random':^28s}")
    print(f"{'':20s} {'error (rmse)':>13s} {'mean error':>13s}   {'error (rmse)':>13s} {'mean error':>13s}")
    rmse = {}
    for name in STRATEGIES[1:]:
        M = imputed[name]
        assert np.array_equal(M[seen], w.observed[seen]), f"{name} changed a reported value"
        assert not np.isnan(M).any(), f"{name} left a hole"
        cells = []
        for mask in (w.lost_low, w.lost_random):
            err = (M - w.truth)[mask]
            cells += [math.sqrt(float((err ** 2).mean())), float(err.mean())]
        rmse[name] = cells
        print(f"{name:20s} {cells[0]:13.2f} {cells[1]:+13.2f}   {cells[2]:13.2f} {cells[3]:+13.2f}")
    print("error = imputed value minus the value the cell would have had, in log2 units; a positive "
          "mean error means the fills are too high.")
    best_low = min(rmse, key=lambda n: rmse[n][0])
    best_random = min(rmse, key=lambda n: rmse[n][2])
    print(f"smallest error on cells lost because low: {best_low}; on cells lost at random: {best_random}. "
          + ("The same strategy is closest for both kinds of loss here." if best_low == best_random
             else "The closest strategy differs with the kind of loss, so no single one is the most accurate."))


_demo("the imputers", _the_imputers, needs=_UPTO[8])

## 13. The result: one test, one promised rate, three ways of filling the holes

Everything is in place. The cell below takes the same matrix, the same test (your `welch_test`)
and the same promised rate, and changes only how the holes are treated. Its rows:

- **complete data**: the answer-key matrix with nothing lost, the ceiling from section 7;
- **no imputation**: your test on the reported values only, so a protein with fewer than two
  reported values in a condition is not tested at all;
- **minimum**, **nearest neighbours**, **left tail**: your three imputations, then the same test.

`switch found` counts the switch proteins (on in one condition, off in the other) among the calls.
A switch protein is the extreme case of a value that is missing because it is low.

In [ ]:
def analyse_world(world: World, seed: int, correct_batch: bool = False) -> dict[str, Calls]:
    """Given to you. Every strategy, then your differential_calls; optionally your batch_correct first.

    "no imputation" is not corrected: `batch_correct` needs a complete matrix.
    """
    results = {"complete data": differential_calls(
        batch_correct(world.truth, world.batch) if correct_batch else world.truth, world.is_case, ALPHA)}
    for name, M in impute_all(world, seed).items():
        if correct_batch and name != "no imputation":
            M = batch_correct(M, world.batch)
        elif correct_batch:
            continue
        results[name] = differential_calls(M, world.is_case, ALPHA)
    return results


def print_call_table(world: World, results: dict[str, Calls], title: str) -> None:
    """Given to you. One row per call set: how many, how many right, how many wrong, and against the promise."""
    print(title)
    print(f"{'':20s} {'tested':>6s} {'called':>6s} {'true':>5s} {'false':>5s} {'fdp':>7s} "
          f"{'missed':>6s} {'switch found':>13s}")
    for name, r in results.items():
        s = score_calls(r.called, world.is_de)
        sw = int((r.called & world.is_switch).sum())
        flag = "  above promised" if s.fdp > ALPHA else ""
        print(f"{name:20s} {int((~np.isnan(r.p)).sum()):6d} {s.called:6d} {s.true_calls:5d} {s.false_calls:5d} "
              f"{100 * s.fdp:6.1f}% {s.missed:6d} {sw:6d} of {int(world.is_switch.sum()):<3d}{flag}")


def _the_result() -> None:
    results = analyse_world(WORLD, SEED)
    _PIPE["results"] = results
    print_call_table(WORLD, results, f"balanced design, promised false discovery rate {ALPHA}, no batch correction")
    imputers = list(STRATEGIES[1:])
    sizes = {n: int(results[n].called.sum()) for n in imputers}
    big, small = max(sizes, key=sizes.get), min(sizes, key=sizes.get)
    print(f"\nthe three imputations call between {sizes[small]} ({small}) and {sizes[big]} ({big}) "
          f"proteins from the same matrix at the same promised rate: a factor of "
          f"{sizes[big] / max(sizes[small], 1):.2f}.")
    over = [n for n in results if score_calls(results[n].called, WORLD.is_de).fdp > ALPHA]
    print("call sets whose false discovery proportion is above the promised rate: "
          + (", ".join(over) if over else "none") + ".")
    every = np.stack([results[n].called for n in imputers])
    truly = WORLD.is_de
    print(f"truly differing proteins called by all three imputations: {int(every[:, truly].all(axis=0).sum())}; "
          f"by exactly one: {int((every[:, truly].sum(axis=0) == 1).sum())}; "
          f"by none of them: {int((~every[:, truly].any(axis=0)).sum())}")


_PIPE: dict[str, Any] = {}
_demo("the result", _the_result, needs=_UPTO[8])

In [ ]:
def _the_overlap() -> None:
    results = _PIPE["results"]
    names = list(STRATEGIES[1:])
    print(f"{'call set A':20s} {'call set B':20s} {'only A':>7s} {'both':>6s} {'only B':>7s}")
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            only_a, both, only_b = overlap_counts(results[names[i]].called, results[names[j]].called)
            print(f"{names[i]:20s} {names[j]:20s} {only_a:7d} {both:6d} {only_b:7d}")
    only_a, both, only_b = overlap_counts(results["no imputation"].called, results["nearest neighbours"].called)
    print(f"{'no imputation':20s} {'nearest neighbours':20s} {only_a:7d} {both:6d} {only_b:7d}")
    print("\nA list is a set of proteins, not a number: two lists of similar size can share few members, "
          "and the columns 'only A' and 'only B' are the proteins whose fate the choice of imputation decided.")
    truth = WORLD.is_de
    for a, b in ((names[0], names[1]), (names[0], names[2]), (names[1], names[2])):
        only_a = results[a].called & ~results[b].called
        only_b = results[b].called & ~results[a].called
        print(f"  called by {a} but not {b}: {int(only_a.sum())}, of which {int((only_a & truth).sum())} truly differ; "
              f"the other way round: {int(only_b.sum())}, of which {int((only_b & truth).sum())} truly differ")


_demo("the overlap", _the_overlap, needs=_UPTO[8])

In [ ]:
def _plot_the_result() -> None:
    results = _PIPE["results"]
    names = list(results)
    true = [score_calls(results[n].called, WORLD.is_de).true_calls for n in names]
    false = [score_calls(results[n].called, WORLD.is_de).false_calls for n in names]
    fig, ax = plt.subplots(figsize=(7, 3.4))
    x = np.arange(len(names))
    ax.bar(x, true, label="truly differ")
    ax.bar(x, false, bottom=true, label="do not differ (false calls)")
    ax.axhline(int(WORLD.is_de.sum()), color="k", linestyle=":", label="planted truth")
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=15, fontsize=8)
    ax.set_ylabel("proteins called")
    ax.legend(fontsize=8)
    fig.tight_layout()
    _show(fig)


_demo("the result plot", _plot_the_result, needs=_UPTO[8])

Read the table before the next paragraph. Three questions, in this order. How different are the list
sizes, and does the difference come from the true column, the false column, or both? Which strategy
found the switch proteins, and which did not, and does that follow from what each one does to a hole
that is low? Which lists are above the promise, and which of them are also the largest?

Here is a reading to test against the tables, not a conclusion. A strategy that fills a hole from
*other proteins* pulls it towards what the typical protein looks like, which for a switch protein
is the wrong direction; a strategy that fills a hole with a low value pulls every hole down, including
the ones lost by chance in proteins that are not low. The error table in section 12 splits the two
kinds of loss, and its mean-error column is where the reading can be checked, and where it could
fail on a world with a different mix of losses. Neither strategy is wrong in general. Which one is
appropriate depends on which kind of loss you have, which is the point of the paper this lesson
starts from (`claims.yaml`).

## 14. Exercise 9 — `batch_correct()`

Batch is the second half of the problem. The rule here is a **location-scale** correction: for
each protein separately, make every batch have the same mean and the same spread.

- each batch `b` has a mean `m_b` and a standard deviation `s_b` (dividing by n−1) for that
  protein;
- the protein has one overall mean `M` (the mean over *all* samples, before correction) and one
  pooled standard deviation `S`: the square root of the sum of the batches' squared deviations
  from their own means, divided by the sum of (n_b − 1);
- a value `x` in batch `b` becomes `M + (x − m_b) * S / s_b`.

If a protein is constant within a batch (`s_b` is exactly zero), that batch is shifted to the
overall mean but not rescaled. The function needs a complete matrix (impute first: it raises
`ValueError` on a NaN) and at least two samples in every batch. The order in which the batches
are listed must not matter.

This is deliberately the plain version, which knows nothing about the condition. Section 15 measures
what that costs when batch and condition are correlated.

<details><summary>💡 Hint 1 — what to think about</summary>

Everything is per protein, so per row, and per batch, so per group of columns. What would you
need to know about each batch first, and what about all batches together, before you can rewrite
any value? Notice that the pooled spread uses the batches' sums of squares, not the average of their
standard deviations.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Get the overall mean of each row. For each batch, get the mean of each row and the sum of squared
deviations from it. Add the sums over batches, divide by the total of (size − 1), take the root:
that is the pooled spread. Then for each batch compute the scale factor per row (one where the
batch's own spread is zero) and write the corrected columns. Use the unique labels, whatever they
are, so that a batch called "b" works as well as a batch called 1.
</details>

In [ ]:
def batch_correct(X: np.ndarray, batch: np.ndarray) -> np.ndarray:
    """Location-scale batch correction, protein by protein, on a complete matrix.

    For each row: `M` = mean over all columns; for each batch, `m_b` and `s_b` (mean and standard
    deviation with n - 1); `S` = sqrt(sum_b sum((x - m_b)^2) / sum_b (n_b - 1)). A value in batch b
    becomes `M + (x - m_b) * S / s_b`, or `M + (x - m_b)` when `s_b` is exactly 0. `batch` may hold
    any labels, in any order.

    Raise `ValueError` if `X` holds a NaN, if `batch` does not have one entry per column, or if
    any batch has fewer than two samples.

    Example:
        >>> X = np.array([[1.0, 3.0, 10.0, 14.0]])
        >>> batch_correct(X, np.array([0, 0, 1, 1])).round(4).tolist()
        [[5.4189, 8.5811, 5.4189, 8.5811]]

    Returns:
        A new float array of the same shape.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_batch_correct() -> None:
    got = batch_correct(np.array([[1.0, 3.0, 10.0, 14.0]]), np.array([0, 0, 1, 1]))
    want = [[5.4189, 8.5811, 5.4189, 8.5811]]
    assert np.allclose(got, want, atol=1e-4), (
        f"expected {want}, got {np.round(got, 4).tolist()}. Each batch is centred on its own mean, rescaled by "
        "pooled spread / its own spread, then moved to the OVERALL mean of the protein")
    got = batch_correct(np.array([[2.0, 2.0, 4.0, 6.0]]), np.array([0, 0, 1, 1]))
    want = [[3.5, 3.5, 3.5 - math.sqrt(0.5), 3.5 + math.sqrt(0.5)]]
    assert np.allclose(got, want), (
        f"a batch in which the protein is constant is shifted to the overall mean but not rescaled (the other batch "
        f"is rescaled by pooled spread / its own spread = 1 / sqrt(2)): expected {np.round(want, 4).tolist()}, "
        f"got {np.round(got, 4).tolist()}")
    X = np.array([[1.0, 8.0, 3.0, 11.0, 2.5, 9.5], [5.0, 1.0, 9.0, 4.0, 2.0, 7.0]])
    a = batch_correct(X, np.array([0, 1, 0, 1, 0, 1]))
    b = batch_correct(X, np.array(["y", "x", "y", "x", "y", "x"]))
    assert np.allclose(a, b), "the answer must not depend on how the batches are named or ordered"
    for lab in (0, 1):
        cols = np.array([0, 1, 0, 1, 0, 1]) == lab
        assert np.allclose(a[:, cols].mean(axis=1), X.mean(axis=1)), (
            "after correction every batch has the protein's overall mean")
    assert np.allclose(batch_correct(X, np.zeros(6, dtype=int)), X), "a single batch is left as it is"
    for bad_x, bad_b in ((np.array([[1.0, np.nan, 3.0, 4.0]]), np.array([0, 0, 1, 1])),
                         (np.ones((1, 4)), np.array([0, 0, 0, 1])), (np.ones((1, 4)), np.array([0, 1]))):
        try:
            batch_correct(bad_x, bad_b)
        except ValueError:
            pass
        else:
            raise AssertionError("a NaN, a batch of one sample, or labels of the wrong length must raise ValueError")
    print("exercise 9 looks right — every batch brought to the same mean and spread")

In [ ]:
_try("exercise 9", _check_batch_correct)

## 15. The result, again: with and without batch correction, in both designs

The same call sets are now built twice per design: on the imputed matrix as it stands, and
on the imputed matrix after your `batch_correct`. Two designs: the balanced one, where each
condition is spread evenly across the batches, and the correlated one, where most controls are in
one batch and most cases in the other. The row "complete data" has no missing values at all, so
whatever it shows is batch and nothing else.

In [ ]:
def _batch_condition_correlation(w: World) -> float:
    """Correlation between the batch label and the condition label, computed without linear algebra."""
    a, b = w.batch.astype(float), w.is_case.astype(float)
    a, b = a - a.mean(), b - b.mean()
    return float((a * b).sum() / math.sqrt(float((a * a).sum() * (b * b).sum())))


def _the_batch_results() -> None:
    for design, world in (("balanced design", WORLD), ("correlated design", CONFOUNDED)):
        print(f"{design}: correlation between batch and condition = {_batch_condition_correlation(world):+.2f}")
        raw = analyse_world(world, SEED, correct_batch=False)
        fixed = analyse_world(world, SEED, correct_batch=True)
        _PIPE[design] = (raw, fixed)
        print_call_table(world, raw, "  as it stands")
        print_call_table(world, fixed, "  after batch correction (no imputation is not corrected: it has holes)")
        c_raw = score_calls(raw["complete data"].called, world.is_de)
        c_fix = score_calls(fixed["complete data"].called, world.is_de)
        print(f"  on the complete matrix the correction changes false calls {c_raw.false_calls} -> "
              f"{c_fix.false_calls} and true calls {c_raw.true_calls} -> {c_fix.true_calls}")
        graded = world.is_de & ~world.is_switch
        for label, res in (("as it stands", raw["complete data"]), ("corrected", fixed["complete data"])):
            ratio = float(np.median(res.log2fc[graded] / world.log2fc[graded]))
            print(f"  complete matrix, {label}: the median estimated fold change is {ratio:.2f} times the planted one "
                  "(moderate differences only)")
        null = ~world.is_de
        shares = [float((res.p[null] < ALPHA).mean()) for res in (raw["complete data"], fixed["complete data"])]
        print(f"  complete matrix: share of the planted non-differing proteins with p < {ALPHA:g}: "
              f"{100 * shares[0]:.1f}% as it stands, {100 * shares[1]:.1f}% corrected (a test whose p-values are "
              f"valid gives about {100 * ALPHA:g}%)")
        print()


_demo("batch results", _the_batch_results, needs=_UPTO[9])

Compare the two designs row by row, in five steps. First the row "complete data" as it stands:
with nothing missing, does the false discovery proportion in the correlated design stay where it
was promised? Then what the correction does to the false calls and to the true calls in each
design; the lists are compared against the planted truth, so a correction that empties the false
column by also emptying the true column would show. Third, the two lines about fold changes: the
correction here knows only the batch label, so it removes whatever differs between batches,
including any part of the condition difference that rides along with the batch. Read what the
ratio does in each design. Fourth, the line about the planted non-differing proteins: it asks the
test itself whether its p-values mean what they say, before any list is made. In the balanced
design each condition holds the same mix of batches, so the batch shift cancels from the difference
of the means but still adds to the spread within each condition. The test counts it as noise, so
its p-values tend to come out too large; removing the shift removes that slack, and the corrected
lists grow. Read the corrected share against the promised rate before you blame the correction for
the false calls of a corrected list. Finally the correlation line above each block: it is what the
correction has to work with.

Batch effects become a major problem when they are correlated with an outcome of interest
(`claims.yaml`). A 2026 study of a large urine peptidomics cohort combined three imputations with
three batch corrections and reported that a correction that ignored disease status removed a large
share of the disease-associated peptides, while one that included it largely preserved the signal
(`claims.yaml`). Including the condition is not free either: when the groups are spread unevenly
over the batches, a correction that keeps the group difference may induce incorrect group
differences downstream and exaggerate the confidence in them (`claims.yaml`), which is why the
correction here stays blind to the condition. Your correction is the kind that ignores the
condition, and the more completely the two are confounded the less a correction that sees only the
batch label can do. The same 2026 study found that the choice of imputation had minimal impact on
its final list, given its missingness structure and normalisation (`claims.yaml`): whether the
imputation matters on a real data set is something to measure there, not to carry over from a
simulated world like this one.

## 16. How much of that was the luck of one seed?

Everything above is one simulated world per design. The cell below builds several more from the
seeds after it, runs every strategy in every design, with and without correction, and
summarises. It prints the range across worlds, not an average alone, and counts the worlds in
which each list ended above the promised rate. Nothing is chosen: the seeds are the next ones in
a row.

In [ ]:
def _the_spread() -> None:
    table: dict[tuple, list] = {}
    for r in range(N_REPLICATES):
        seed = SEED + r
        for design, confounded in (("balanced design", False), ("correlated design", True)):
            world = keep_observed(generate_world(seed, confounded))
            imputed = impute_all(world, seed)
            for corrected in (False, True):
                results = {"complete data": differential_calls(
                    batch_correct(world.truth, world.batch) if corrected else world.truth, world.is_case, ALPHA)}
                for name, M in imputed.items():
                    if corrected and name == "no imputation":
                        continue
                    results[name] = differential_calls(batch_correct(M, world.batch) if corrected else M,
                                                       world.is_case, ALPHA)
                for name, res in results.items():
                    s = score_calls(res.called, world.is_de)
                    table.setdefault((design, corrected, name), []).append(
                        (s.called, s.false_calls, s.fdp, int((res.called & world.is_switch).sum()),
                         int(world.is_switch.sum())))
    _PIPE["spread"] = table
    for design in ("balanced design", "correlated design"):
        print(f"{design}: {N_REPLICATES} worlds, promised false discovery rate {ALPHA}")
        print(f"  {'':34s} {'called: mean [min-max]':>24s} {'false discovery proportion':>29s} "
              f"{'worlds above':>13s} {'switch found':>13s}")
        for (d, corrected, name), rows in table.items():
            if d != design:
                continue
            called = [x[0] for x in rows]
            fdp = [100 * x[2] for x in rows]
            above = sum(x[2] > ALPHA for x in rows)
            found = sum(x[3] for x in rows) / sum(x[4] for x in rows)
            label = f"{name}{', corrected' if corrected else ''}"
            print(f"  {label:34s} {np.mean(called):8.1f} [{min(called):4d}-{max(called):4d}] "
                  f"{np.mean(fdp):8.1f}% mean, {max(fdp):5.1f}% max {above:9d} of {N_REPLICATES} "
                  f"{100 * found:12.1f}%")
        print()
    counts = {n: 0 for n in STRATEGIES[1:]}
    for k in range(N_REPLICATES):
        best = max(STRATEGIES[1:], key=lambda n: table[("balanced design", False, n)][k][0])
        counts[best] += 1
    print("balanced design, no correction: the imputation with the most calls in each world: "
          + ", ".join(f"{n} in {c}" for n, c in counts.items()) + f" of {N_REPLICATES} worlds")
    ratios = []
    for k in range(N_REPLICATES):
        sizes = [table[("balanced design", False, n)][k][0] for n in STRATEGIES[1:]]
        ratios.append(max(sizes) / max(min(sizes), 1))
    larger = sum(x > ratios[0] for x in ratios[1:])
    print(f"largest imputation list over smallest, balanced design, no correction: from {min(ratios):.2f} to "
          f"{max(ratios):.2f} across the worlds; the world studied in sections 2 to 15 (seed {SEED}) gives "
          f"{ratios[0]:.2f}, and {larger} of the other {N_REPLICATES - 1} worlds give a larger one")


_demo("the spread", _the_spread, needs=_UPTO[9], limit=170)

Read the column "worlds above" as the honest version of the promised rate. The promise is about
the long-run average, so a single list can sit above it without any bug; the lists that sit above
it in most worlds are telling you something about the method, not about luck. The column "switch
found" shows whether the loss of the on/off proteins under a strategy is a feature of one world or
of the strategy. And the range in "called" is how much of any single number in the tables above you
should expect to change with the seed. The last line places the world you studied in detail among
the others: if it sits at one end of the range, the single-world factor printed in section 13
overstates or understates how much the imputation matters in a typical world.

## 17. Common mistakes

- **Reporting which imputation was used, not what it changed.** A methods line that names the
  imputation says nothing about how many calls depend on it; sections 13 and 16 count them.
- **Treating the imputed values as measurements.** The test cannot tell a filled value from a
  measured one, so it under-states the uncertainty, and a promised rate that assumes valid p-values
  is then only an approximation. Sections 15 and 16 count the lists that end up above the promise.
- **Choosing the imputation before asking what the holes are.** Section 8 shows the symptom of
  values lost because they were low. Whether your holes are like that is the question, and a
  method that fits one kind can be wrong for the other.
- **Counting untestable proteins as tests.** They were never tested; see the cell below.
- **Correcting for batch when batch is correlated with the condition, and calling the result
  clean.** The counts in section 15 include what was lost.
- **Ranking the strategies on one seed.** Section 16 is there to remove that temptation.
- **Quoting an adjusted threshold without the level.** The false discovery rate is promised for
  a list, and the list depends on every choice above it.

In [ ]:
def _why_untestable_are_not_tests() -> None:
    res = welch_test(WORLD.observed, WORLD.is_case)
    testable = ~np.isnan(res.p)
    right, _ = benjamini_hochberg(res.p[testable], ALPHA)
    padded, _ = benjamini_hochberg(np.where(testable, res.p, 1.0), ALPHA)
    padded = padded[testable]
    n_right, n_padded = int((right <= ALPHA).sum()), int((padded <= ALPHA).sum())
    print(f"{int((~testable).sum())} of {len(testable)} proteins are untestable without imputation")
    print(f"adjusting only the {int(testable.sum())} tests that were run: {n_right} called; "
          f"counting the untestable ones as p = 1: {n_padded} called ({n_padded - n_right:+d})")
    grown = right > 0
    print(f"the q-values grow by a factor of up to {float((padded[grown] / right[grown]).max()):.3f} "
          f"(the ratio of the two test counts is {len(testable) / int(testable.sum()):.3f}): "
          + ("here that cost the list members, and a list with more proteins near the threshold loses more."
             if n_padded < n_right else
             "here no protein sat close enough to the threshold to drop out, but a list with proteins just "
             "below it would lose them."))


_demo("untestable are not tests", _why_untestable_are_not_tests, needs=_UPTO[2])

## 18. Self-check

1. A protein has no reported value in any control sample, and is reported in every case sample at
   a level near the detection limit. What is the most reasonable reading of the missing values?
   - (a) they were lost at random, so any imputation is as good as another
   - (b) the instrument failed on the controls, so those samples should be dropped
   - (c) they were probably lost because the protein was low in the controls, so the holes carry
     information about its level

2. In the Benjamini-Hochberg procedure, a protein that could not be tested is given a p-value of 1
   and counted among the tests. Compared with leaving it out, the other proteins' q-values are:
   - (a) larger, because the number of tests has grown, so the list can only shrink or stay the same
   - (b) unchanged, because a p-value of 1 is never called
   - (c) smaller, because there is more evidence against the null overall

3. A call list is accepted at a promised false discovery rate, and the false discovery proportion
   against the answer key is above it. Which is a reason that needs no bug at all?
   - (a) Benjamini-Hochberg only works on data without missing values
   - (b) the promise is about an expectation, and imputed values are treated by the test as if they
     had been measured, so its p-values are somewhat too small
   - (c) the promised rate is compared with the number of calls, not their share

4. Batch is correlated with the condition, and you apply a correction that knows only the batch.
   What can happen?
   - (a) nothing that matters: the correction removes technical variation and only that
   - (b) the false calls can disappear and real differences with them, because the two cannot be
     separated from the batch labels alone
   - (c) the correction makes the batch effect larger

5. Which imputation should you use?
   - (a) whichever has the lowest error on the cells that were lost, in one simulated world
   - (b) always the simplest, because the test result does not depend on it
   - (c) the one that suits the missing values you have, which you check with the symptom in
     section 8 and with the call sets, not only with an error

Answers, with the reasoning, come with this lesson's worked solution when you enrol on Synapsa.

In [ ]:
def _methods_statement() -> None:
    res = _PIPE["results"]
    print("METHODS — a differential-abundance list without these lines cannot be reproduced or compared")
    print(f"  matrix            {WORLD.observed.shape[0]} proteins x {WORLD.observed.shape[1]} samples, log2 scale; "
          f"proteins reported in fewer than {MIN_OBSERVED} samples removed")
    print(f"  missing values    {100 * np.isnan(WORLD.observed).mean():.1f}% of cells; imputation: one of "
          f"minimum per sample, k = {K_NEIGHBOURS} nearest proteins (at least {MIN_COMMON} shared samples), "
          f"normal fitted to the lowest {100 * TAIL:g}% of each sample; or none")
    print("  test              Welch t-test on reported values (untestable proteins not tested), two-sided")
    print(f"  error control     Benjamini-Hochberg, promised rate {ALPHA:g}, applied to the tested proteins only")
    print("  batch             location-scale correction per protein and batch, on the imputed matrix, "
          "unaware of the condition")
    sizes = ", ".join(f"{n} {int(res[n].called.sum())}" for n in STRATEGIES)
    print(f"  proteins called   {sizes} (balanced design, no correction)")
    lo, hi = (f(int(res[n].called.sum()) for n in STRATEGIES) for f in (min, max))
    print(f"  the choices {'changed' if hi > lo else 'did not change'} the size of the list: the same matrix and the "
          f"same promised rate give call sets of {lo} to {hi} proteins (section 13).")


_demo("methods statement", _methods_statement, needs=_UPTO[9])

## What you built, and where it goes next

You now have the whole path from a matrix with holes to a call list: a test, an error control, three
ways of filling the holes, a batch correction, and the counts against an answer key that show what
each choice did. The habit to carry forward is the line in the methods statement above: **a
differential-abundance list is a claim about a set of proteins, and the set depends on every choice
made before the test.** The capstone of this programme runs this quantitation step on a generated
cohort, together with the leakage audit and the integration.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_bh),
                              ("exercise 2", _check_welch),
                              ("exercise 3", _check_differential_calls),
                              ("exercise 4", _check_scoring),
                              ("exercise 5", _check_missingness),
                              ("exercise 6", _check_minimum),
                              ("exercise 7", _check_knn),
                              ("exercise 8", _check_left_tail),
                              ("exercise 9", _check_batch_correct)):
            _try(_name, _check)
    _progress_board()
    print(f"\nnotebook wall time so far: {time.perf_counter() - _LESSON_T0:.1f}s")
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.